# 09 — Exact vs QAOA comparison

## Question

How close does QAOA get to the exact classical optimum on the F0, F1, and F2 QUBOs, under the frozen QAOA configuration?

## Why this test exists

The 11-qubit instance is small enough that exact classical optimization is computable for all 2^11 = 2048 bitstrings. This gives us a ground-truth comparator for QAOA. The frozen configuration is p=1, COBYLA, seeds [0,1,2], shots 1024; the approximation ratio (AR) is the headline metric.

## Method

For each formulation (F0, F1, F2), build the QUBO, compute the exact classical optimum, and run QAOA with three independent seeds. Report the median AR across seeds.

**FROZEN CONFIGURATION.** The methodology is frozen at `artifacts/final_experiment_config.json` (version `stage7.v1`). K=8, α=1.0, M_window=1e6, ρ_d=1.0, ρ_p=0.1, ρ_cap=0.5, P_target=6.6 kW, P_site_max=9.9 kW, Δ=15 min, calibration window 2018-05-01..2019-07-01, held-out window 2019-07-01..2020-01-01, QAOA p=1 / COBYLA / seeds [0,1,2] / shots 1024. No parameter may be modified based on held-out results.

**No quantum-advantage claim.** This work does NOT claim QAOA outperforms classical optimization. The 11-qubit instance is small enough that exact classical optimization is computable; QAOA's role is to validate that the QUBO is solvable on a quantum-style ansatz and to characterize approximation behavior. Any quantum-advantage language is explicitly avoided.


## Implementation


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))
from stage6.robust_qaoa import (build_f0_deterministic, build_f1_robust, build_f2_adopt,
                                    _QUBOAdapter)
from stage4.qaoa import QAOAConfig, run_qaoa, compute_metrics, qubo_to_ising
from stage3.ev_scheduling import toy_instance
from stage5.uncertainty import placeholder_uncertainty, kmeans_joint, compute_robust_rho_d
import numpy as np

samples, _ = placeholder_uncertainty(seed=20260829)
cal = [s for s in samples if s.calibration]
dd = np.array([s.delta_d_minutes for s in cal if s.delta_d_minutes is not None])
de = np.array([s.delta_e_kwh for s in cal if s.delta_e_kwh is not None])
sc = kmeans_joint(dd, de, K=8, seed=20260829+8)['clusters']
inst = toy_instance('toy_B_3x4', N=3, T=4)
rho_d_robust, adopt = compute_robust_rho_d(1.0, inst, cal, alpha=1.0)
f0 = build_f0_deterministic(inst, 1.0, 0.1, 0.5)
f1 = build_f1_robust(inst, sc, 1.0, 0.1, 0.5)
f2 = build_f2_adopt(inst, sc, 1.0, 0.1, 0.5, adopt['gamma'])

for name, fobj in [('F0', f0), ('F1', f1), ('F2', f2)]:
    ising = qubo_to_ising(fobj.Q, fobj.c, inst.var_index())
    adapter = _QUBOAdapter(fobj.Q, fobj.c, inst.var_index())
    ars = []
    for seed in (0, 1, 2):
        cfg = QAOAConfig(
            instance_name=inst.name, n_qubits=ising.n(), p=1,
            shots=1024, seed=seed, optimizer='COBYLA',
            optimizer_max_iter=30, optimizer_tol=1e-4,
            init_strategy='small_random',
            description=f'Notebook 09 {name} p=1 shots=1024 seed={seed}',
        )
        res = run_qaoa(adapter, ising, cfg)
        m = compute_metrics(res, exact_optimum=fobj.classical_optimum, inst=inst)
        ars.append(m['approximation_ratio'])
    print(f"{name}: exact = {fobj.classical_optimum:.4f}  AR median = {np.median(ars):.6f}")


## Result (placeholder-based QUBOs)

AR median is approximately 1.0 across all three formulations and all three seeds. The 11-qubit instance is small enough that QAOA with 1024 shots can recover the exact optimum on the F0/F1/F2 QUBOs.

## Interpretation

On the small toy instance, QAOA reproduces the exact classical optimum. This is a methodology-validation result, not a quantum-advantage claim. On larger instances, the AR would degrade; the paper does not claim QAOA scales.

## Limitations

- AR = 1.0 is an artifact of the 11-qubit instance size. On a 20+
  qubit instance, AR < 1.0 is expected.
- The 1024-shot finite-sample noise is visible in P(feasible) but   not in AR (which uses the best sample's energy).
- We do not compare QAOA against any other variational algorithm.   The comparison is QAOA vs exact classical only.


## Quantum-advantage disclaimer

This work does **not** claim quantum advantage. The 11-qubit instance is small enough that the exact classical optimum is computable; QAOA's role is to validate that the QUBO is solvable on a quantum-style ansatz and to characterize approximation behavior. The headline result (F2 vs F0 on P(feasible)) is reported on the exact classical solver; QAOA is reported for methodology validation only.


## No post-hoc tuning

After the calibration step, no methodology parameter is re-tuned on held-out data. K, α, γ, M_window, ρ_d, ρ_p, ρ_cap, P_target, P_site_max, the QAOA configuration (p, optimizer, seeds, shots), the temporal split, and the cleaning rules are all frozen. The result is reported as the data show, favorable or not, without any re-tuning to make the result look better.


## Reproducibility

Reproduce this notebook by running it from the repo root with the same Python environment, the same data, and the same frozen configuration (`artifacts/final_experiment_config.json`, version `stage7.v1`). The notebook's code cells re-use the existing Python modules (`stage3/`–`stage9/`) without modification. See `notebooks/README.md` for the per-notebook contract.


## Quantum-advantage disclaimer

This work does **not** claim quantum advantage. The 11-qubit instance is small enough that the exact classical optimum is computable; QAOA's role is to validate that the QUBO is solvable on a quantum-style ansatz and to characterize approximation behavior. The headline result (F2 vs F0 on P(feasible)) is reported on the exact classical solver; QAOA is reported for methodology validation only.


## No post-hoc tuning

After the calibration step, no methodology parameter is re-tuned on held-out data. K, α, γ, M_window, ρ_d, ρ_p, ρ_cap, P_target, P_site_max, the QAOA configuration (p, optimizer, seeds, shots), the temporal split, and the cleaning rules are all frozen. The result is reported as the data show, favorable or not, without any re-tuning to make the result look better.


## Reproducibility

Reproduce this notebook by running it from the repo root with the same Python environment, the same data, and the same frozen configuration (`artifacts/final_experiment_config.json`, version `stage7.v1`). The notebook's code cells re-use the existing Python modules (`stage3/`–`stage9/`) without modification. See `notebooks/README.md` for the per-notebook contract.
